In [ ]:
pip install roboflow ultralytics opencv-python numpy

In [ ]:
"""
YOLOv8 Segmentation Pipeline
----------------------------
Before running this, install the required libraries in your terminal:
pip install ultralytics opencv-python numpy
"""

import cv2
import numpy as np
from ultralytics import YOLO

def main():
    print("Loading YOLOv8 model...")
    model = YOLO('yolov8n-seg.pt')

    image_path = '/content/diet_coke_01.jpg'

    original_image = cv2.imread(image_path)

    if original_image is None:
        print(f"Error: Could not load image at {image_path}")
        return

    original_height, original_width = original_image.shape[:2]

    print("Running inference...")
    results = model.predict(source=original_image, save=False)

    result = results[0]

    if result.masks is None:
        print("No objects were detected.")
        return

    mask_tensor = result.masks.data
    masks_numpy = mask_tensor.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy()

    combined_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    print(f"Detected {len(masks_numpy)} objects. Processing masks...")

    for i in range(len(masks_numpy)):
        single_mask = masks_numpy[i]
        resized_mask = cv2.resize(single_mask, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
        binary_mask = (resized_mask * 255).astype(np.uint8)
        combined_mask = cv2.bitwise_or(combined_mask, binary_mask)

    mask_3_channel = cv2.cvtColor(combined_mask, cv2.COLOR_GRAY2BGR)
    isolated_objects = cv2.bitwise_and(original_image, mask_3_channel)

    output_filename = "segmented_output.jpg"
    cv2.imwrite(output_filename, isolated_objects)
    print(f"Success! Segmented image saved as {output_filename}")

    # cv2.imshow("Isolated Objects", isolated_objects)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Loading YOLOv8 model...
Running inference...

0: 480x640 1 person, 1 chair, 1 oven, 1 book, 12.0ms
Speed: 7.1ms preprocess, 12.0ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)
Detected 4 objects. Processing masks...
Success! Segmented image saved as segmented_output.jpg


In [ ]:
"""
Roboflow YOLOv8 Training & Robotic Perception Inference Pipeline
-----------------------------------------------------------------
This script automatically:
1. Downloads your custom segmentation dataset from Roboflow.
2. Trains a custom YOLOv8-Seg model using your dataset.
3. Performs inference on the validation set using your newly trained model (`best.pt`).
4. Extracts segmentation masks and applies advanced post-processing:
   - Centroid calculation via raw image moments.
   - Orientation angle estimation via central moments.
   - Optimal grasp center selection via Euclidean Distance Transform.
5. Saves visual diagnostic dashboards with annotated grasp coordinates.
"""

import os
import glob
import cv2
import numpy as np
import torch
from roboflow import Roboflow
from ultralytics import YOLO

ROBOFLOW_API_KEY = "MBwFWKDX9ZIw9gy7hdNn"
ROBOFLOW_WORKSPACE = "dhairya-prajapati-poezj"
ROBOFLOW_PROJECT = "robotic_arm_grasp_dataset"
ROBOFLOW_VERSION = 2

EPOCHS = 25
BATCH_SIZE = 8
IMAGE_SIZE = 640
PROJECT_NAME = "robotic_grasp"
RUN_NAME = "yolov8_grasp_items_seg_version_2"
OUTPUT_DIR = "valid_results"


def download_roboflow_dataset() -> str:
    print("[ROBOFLOW] Initializing Roboflow client...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    print(f"[ROBOFLOW] Accessing project '{ROBOFLOW_PROJECT}'...")
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

    print(f"[ROBOFLOW] Downloading version {ROBOFLOW_VERSION} in YOLOv8 format...")
    dataset = project.version(ROBOFLOW_VERSION).download("yolov8")

    return dataset.location


def train_yolov8_segmentation(dataset_path: str) -> str:
    dataset_yaml = os.path.join(dataset_path, "data.yaml")

    if not os.path.exists(dataset_yaml):
        raise FileNotFoundError(f"Could not locate data.yaml in {dataset_path}")

    print("\n" + "="*50)
    print("           STARTING YOLOv8-SEG TRAINING LOOP")
    print("="*50)

    model = YOLO("yolov8n-seg.pt")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[TRAIN] Target device identified: {device.upper()}")

    model.train(
        data=dataset_yaml,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=device,
        project=PROJECT_NAME,
        name=RUN_NAME,
        workers=4,
        val=True,
        save=True
    )

    best_weights_path = os.path.join(PROJECT_NAME, RUN_NAME, "weights", "best.pt")

    if not os.path.exists(best_weights_path):
        best_weights_path = os.path.abspath(f"runs/segment/robotic_grasp/{RUN_NAME}/weights/best.pt")

    print(f"[TRAIN] Training finished successfully.")
    print(f"[TRAIN] Saved fine-tuned weights to: {best_weights_path}")
    print("="*50 + "\n")

    return best_weights_path


def calculate_grasp_attributes(binary_mask: np.ndarray) -> tuple:
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None, 0.0, None

    largest_contour = max(contours, key=cv2.contourArea)
    moments = cv2.moments(largest_contour)

    if moments["m00"] == 0:
        return None, 0.0, None

    cx = int(moments["m10"] / moments["m00"])
    cy = int(moments["m01"] / moments["m00"])

    mu20 = moments["mu20"]
    mu02 = moments["mu02"]
    mu11 = moments["mu11"]

    angle_rad = 0.5 * np.arctan2(2 * mu11, mu20 - mu02)
    angle_deg = np.degrees(angle_rad)

    dist_transform = cv2.distanceTransform(binary_mask, cv2.DIST_L2, 5)
    _, _, _, max_loc = cv2.minMaxLoc(dist_transform)
    gx, gy = max_loc

    return (cx, cy), angle_deg, (gx, gy)


def process_validation_set(dataset_path: str, trained_weights_path: str):
    valid_images_dir = os.path.join(dataset_path, "valid", "images")

    if not os.path.exists(valid_images_dir):
        print(f"[ERROR] Could not find validation images directory at: {valid_images_dir}")
        return

    image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    image_paths = []

    for ext in image_extensions:
        image_paths.extend(glob.glob(os.path.join(valid_images_dir, ext)))

    if not image_paths:
        print(f"[ERROR] No validation images discovered at: {valid_images_dir}")
        return

    print(f"[INFERENCE] Found {len(image_paths)} images in the validation set.")
    print(f"[INFERENCE] Loading newly trained custom model from: {trained_weights_path}")

    model = YOLO(trained_weights_path)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"[INFERENCE] Output results will be saved to: '{OUTPUT_DIR}/'")

    for idx, img_path in enumerate(image_paths):
        filename = os.path.basename(img_path)
        print(f"\n[{idx+1}/{len(image_paths)}] Processing: {filename}")

        original_image = cv2.imread(img_path)

        if original_image is None:
            print(f"  [SKIP] Could not load image {filename}")
            continue

        original_height, original_width = original_image.shape[:2]

        results = model.predict(source=original_image, save=False, verbose=False)
        result = results[0]

        if result.masks is None or len(result.masks.data) == 0:
            print("  [WARN] No target masks detected in this image.")
            output_filepath = os.path.join(OUTPUT_DIR, f"no_detect_{filename}")
            cv2.imwrite(output_filepath, original_image)
            continue

        masks_numpy = result.masks.data.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        confidences = result.boxes.conf.cpu().numpy()
        class_names = model.names

        overlay_canvas = original_image.copy()

        np.random.seed(42)
        colors = np.random.randint(0, 255, size=(len(class_names), 3), dtype=np.uint8)
        print(f"  Detected {len(masks_numpy)} instances. Calculating grasping telemetry...")

        for i in range(len(masks_numpy)):
            class_id = classes[i]
            class_name = class_names[class_id]
            confidence = confidences[i]
            color = colors[class_id].tolist()

            single_mask = masks_numpy[i]
            resized_mask = cv2.resize(single_mask, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
            binary_mask = (resized_mask > 0.5).astype(np.uint8) * 255
            centroid, angle, grasp_center = calculate_grasp_attributes(binary_mask)

            if centroid is None or grasp_center is None:
                continue

            color_mask_img = np.zeros_like(original_image)
            color_mask_img[binary_mask > 0] = color
            overlay_canvas[binary_mask > 0] = cv2.addWeighted(overlay_canvas, 0.4, color_mask_img, 0.6, 0)[binary_mask > 0]

            contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay_canvas, contours, -1, color, 2)
            cv2.circle(overlay_canvas, centroid, 4, (255, 255, 255), -1)
            cv2.circle(overlay_canvas, grasp_center, 8, (0, 0, 255), 2)
            cv2.circle(overlay_canvas, grasp_center, 3, (0, 0, 255), -1)

            rad = np.radians(angle)
            length = 40
            x1 = int(grasp_center[0] - length * np.cos(rad))
            y1 = int(grasp_center[1] - length * np.sin(rad))
            x2 = int(grasp_center[0] + length * np.cos(rad))
            y2 = int(grasp_center[1] + length * np.sin(rad))
            cv2.line(overlay_canvas, (x1, y1), (x2, y2), (0, 255, 255), 2)

            label_text = f"{class_name} ({confidence*100:.0f}%)"
            angle_text = f"Angle: {angle:.1f}deg"

            cv2.putText(overlay_canvas, label_text, (grasp_center[0] + 12, grasp_center[1] - 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(overlay_canvas, angle_text, (grasp_center[0] + 12, grasp_center[1] + 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1, cv2.LINE_AA)

        output_filepath = os.path.join(OUTPUT_DIR, f"grasp_annotated_{filename}")
        cv2.imwrite(output_filepath, overlay_canvas)
        print(f"  Processed result saved to {output_filepath}")

    print("\n[SUCCESS] Custom trained pipeline has completely analyzed the validation set!")


def main():
    try:
        dataset_path = download_roboflow_dataset()
        best_weights = train_yolov8_segmentation(dataset_path)
        process_validation_set(dataset_path, best_weights)

    except Exception as e:
        print(f"\n[FATAL ERROR] Pipeline failed: {e}")


if __name__ == "__main__":
    main()

[ROBOFLOW] Initializing Roboflow client...
[ROBOFLOW] Accessing project 'robotic_arm_grasp_dataset'...
loading Roboflow workspace...
loading Roboflow project...
[ROBOFLOW] Downloading version 2 in YOLOv8 format...
Exporting format yolov8 in progress : 60.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to Robotic_Arm_Grasp_Dataset-2 in yolov8:: 100%|██████████| 569/569 [00:00<00:00, 7920.45it/s]



           STARTING YOLOv8-SEG TRAINING LOOP
[TRAIN] Target device identified: CUDA
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Robotic_Arm_Grasp_Dataset-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('runs', 'zip', '/content/runs')

'/content/runs.zip'